# Week 4 · Course 7 · Decision Trees

**Sections**
1. Regression Trees — recursive binary splitting, RSS criterion `(0:00 – 0:25)`
2. Classification Trees — Gini, cross-entropy, error rate `(0:25 – 0:50)`
3. Cost-Complexity Pruning — α parameter, K-fold CV `(0:50 – 1:10)`
4. Trees vs Linear Models — when each wins `(1:10 – 1:20)`
5. Strengths, Weaknesses & Variance Demo `(1:20 – 1:30)`

**Libraries:** `sklearn`, `numpy`, `pandas`, `matplotlib`, `seaborn`

## Setup

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import accuracy_score, mean_squared_error, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from shared.data_utils import load_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
rng = np.random.default_rng(0)

In [ ]:
hitters = load_dataset('hitters')
carseats = load_dataset('carseats')
boston = load_dataset('boston')

# Heart disease — download from ISLR2 repo or simulate
try:
    heart = pd.read_csv('https://www.statlearning.com/s/Heart.csv', index_col=0)
    heart = heart.dropna()
except Exception:
    rng2 = np.random.default_rng(42)
    n = 303
    age   = rng2.integers(29, 77, n)
    chol  = rng2.integers(130, 565, n).astype(float)
    thalach = rng2.integers(71, 202, n).astype(float)
    fbs   = rng2.choice([0, 1], n, p=[0.85, 0.15])
    sex   = rng2.choice(['Male', 'Female'], n)
    cp    = rng2.choice(['typical', 'atypical', 'non-anginal', 'asymptomatic'], n)
    thal  = rng2.choice(['normal', 'fixed', 'reversible'], n)
    logit = (-6 + 0.05*age - 0.003*chol + 0.02*thalach
              + (sex == 'Male').astype(float)
              + (cp == 'asymptomatic').astype(float) * 2)
    prob  = 1 / (1 + np.exp(-logit))
    ahd   = rng2.binomial(1, prob).astype(str)
    ahd[ahd == '1'] = 'Yes'
    ahd[ahd == '0'] = 'No'
    heart = pd.DataFrame({
        'Age': age, 'Sex': sex, 'ChestPain': cp, 'RestBP': rng2.integers(94, 200, n),
        'Chol': chol, 'Fbs': fbs, 'RestECG': rng2.integers(0, 3, n),
        'MaxHR': thalach, 'ExAng': rng2.choice([0, 1], n),
        'Oldpeak': np.clip(rng2.normal(1.0, 1.2, n), 0, 6.2).round(1),
        'Slope': rng2.integers(1, 4, n), 'Ca': rng2.integers(0, 4, n),
        'Thal': thal, 'AHD': ahd
    })

print('hitters:', hitters.shape)
print('carseats:', carseats.shape)
print('boston:', boston.shape)
print('heart:', heart.shape)

---
## 1. Regression Trees

A **regression tree** recursively partitions the feature space into rectangular regions and predicts the **mean response** in each region.

### 1.1 The algorithm (recursive binary splitting)

At each step, choose the feature $X_j$ and cutpoint $s$ that minimizes the total within-region RSS:

$$\min_{j,\, s} \left[
  \sum_{i:\, x_{ij} < s} (y_i - \hat{y}_{R_1})^2 +
  \sum_{i:\, x_{ij} \geq s} (y_i - \hat{y}_{R_2})^2
\right]$$

where $\hat{y}_{R_k}$ is the mean of the training responses in region $k$.  
The search is **greedy** — we pick the best split at this node without lookahead.

**Key insight:** Any single split produces two regions. We then recurse within each region, so the final partition can be complex despite each individual choice being simple.

### 1.2 Baseball salary example (ISLR Figure 8.1)

Predict `log(Salary)` from `Years` (years in the major league) and `Hits` (number of hits in the previous year).  
We first fit a depth-2 tree to reproduce the classic ISLR partition diagram.

In [ ]:
h = hitters.dropna(subset=['Salary']).copy()
h['LogSalary'] = np.log(h['Salary'])

X2 = h[['Years', 'Hits']]
y_salary = h['LogSalary']

# Fit a shallow tree — reproduces the 3-region ISLR diagram
tree2 = DecisionTreeRegressor(max_depth=2, random_state=0)
tree2.fit(X2, y_salary)

fig, ax = plt.subplots(figsize=(10, 5))
plot_tree(tree2, feature_names=['Years', 'Hits'], filled=True, ax=ax,
          fontsize=10, impurity=False, precision=2)
ax.set_title('Depth-2 regression tree: log(Salary) ~ Years + Hits')
plt.show()

print('Leaf predictions (mean log salary):')
for leaf_id, val in enumerate(tree2.tree_.value[:, 0, 0]):
    if tree2.tree_.children_left[leaf_id] == -1:  # leaf
        n = tree2.tree_.n_node_samples[leaf_id]
        print(f'  Leaf {leaf_id}: mean log salary = {val:.3f}  (exp = ${np.exp(val)*1000:.0f}k), n={n}')

In [ ]:
# Partition plot — exactly like ISLR Figure 8.1
years_grid = np.linspace(h['Years'].min() - 0.5, h['Years'].max() + 0.5, 300)
hits_grid  = np.linspace(h['Hits'].min() - 5, h['Hits'].max() + 5, 300)
YY, HH = np.meshgrid(years_grid, hits_grid)
Z = tree2.predict(np.c_[YY.ravel(), HH.ravel()]).reshape(YY.shape)

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(YY, HH, Z, levels=20, cmap='YlOrRd', alpha=0.7)
sc = ax.scatter(h['Years'], h['Hits'], c=h['LogSalary'],
                cmap='YlOrRd', edgecolors='k', linewidths=0.4, s=30)
# Draw split lines
split_years = tree2.tree_.threshold[0]  # root split on Years
ax.axvline(split_years, color='navy', lw=2, label=f'Years < {split_years:.1f}')
# Second split (right child)
right_child = tree2.tree_.children_right[0]
if tree2.tree_.feature[right_child] == 1:  # Hits
    split_hits = tree2.tree_.threshold[right_child]
    ax.plot([split_years, h['Years'].max()+0.5], [split_hits, split_hits],
            color='navy', lw=2, linestyle='--', label=f'Hits < {split_hits:.0f}')
plt.colorbar(sc, ax=ax, label='log(Salary)')
ax.set_xlabel('Years'); ax.set_ylabel('Hits')
ax.set_title('Partition of feature space (depth-2 tree)')
ax.legend(); plt.show()

In [ ]:
# Full tree on all Hitters features (ISLR Table 8.1 approach)
cat_cols = ['League', 'Division', 'NewLeague']
h_enc = pd.get_dummies(h.drop(columns=['Salary']),
                        columns=cat_cols, drop_first=True, dtype=float)
X_all = h_enc.drop(columns=['LogSalary'])
y_all = h_enc['LogSalary']

Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.25, random_state=0)

full_reg = DecisionTreeRegressor(random_state=0)
full_reg.fit(Xtr, ytr)
print(f'Unpruned tree: depth={full_reg.get_depth()}, leaves={full_reg.get_n_leaves()}')
print(f'Train MSE = {mean_squared_error(ytr, full_reg.predict(Xtr)):.4f}')
print(f'Test  MSE = {mean_squared_error(yte, full_reg.predict(Xte)):.4f}')

---
## 2. Classification Trees

Classification trees are built the same way as regression trees, but the split criterion measures **node impurity** instead of RSS.

### 2.1 Impurity measures

For a node with $K$ classes and estimated class proportions $\hat{p}_{mk}$:

| Measure | Formula | Notes |
|---|---|---|
| **Classification error** | $1 - \max_k \hat{p}_{mk}$ | Simple but not sensitive to class balance |
| **Gini index** | $\sum_k \hat{p}_{mk}(1 - \hat{p}_{mk})$ | Measures total variance; preferred for growing |
| **Cross-entropy** | $-\sum_k \hat{p}_{mk}\log\hat{p}_{mk}$ | Information-theoretic; similar results to Gini |

**Why not just use error rate?** Error rate is insensitive to the distribution of errors — a node with 49%/51% split looks just as impure as 1%/99%. Gini and entropy give credit for nodes that are *almost* pure.

In [ ]:
# Visualise all three impurity measures as a function of p (binary case)
p = np.linspace(0.001, 0.999, 300)

gini      = 2 * p * (1 - p)        # = 1 - p² - (1-p)² (binary)
entropy   = -(p*np.log(p) + (1-p)*np.log(1-p))
error_rate = 1 - np.maximum(p, 1-p)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, gini,      label='Gini index',         lw=2)
ax.plot(p, entropy/2, label='Entropy / 2',         lw=2, linestyle='--')
ax.plot(p, error_rate,label='Classification error',lw=2, linestyle=':')
ax.axvline(0.5, color='grey', lw=1, linestyle='--', alpha=0.6)
ax.set_xlabel('p (proportion of class 1)')
ax.set_ylabel('Impurity')
ax.set_title('Node impurity measures (binary classification)')
ax.legend(); plt.show()

print('At p=0.5 (worst case):')
print(f'  Gini      = {2*0.5*0.5:.3f}')
print(f'  Entropy/2 = {-np.log(0.5)/2:.3f}')
print(f'  Error     = {0.5:.3f}')

### 2.2 Carseats — predict `High` sales

In [ ]:
c = carseats.copy()
c['High'] = (c['Sales'] > 8).astype(int)
c = pd.get_dummies(c.drop(columns=['Sales']),
                   columns=['ShelveLoc', 'Urban', 'US'], drop_first=True, dtype=float)
X_cs = c.drop(columns=['High']); y_cs = c['High']
Xtr_cs, Xte_cs, ytr_cs, yte_cs = train_test_split(
    X_cs, y_cs, test_size=0.3, random_state=0, stratify=y_cs)

# Fit full unpruned and a depth-3 tree
tree_full = DecisionTreeClassifier(random_state=0).fit(Xtr_cs, ytr_cs)
tree_d3   = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr_cs, ytr_cs)

print(f'Full tree  — depth={tree_full.get_depth()}, '
      f'train acc={tree_full.score(Xtr_cs, ytr_cs):.4f}, '
      f'test acc={tree_full.score(Xte_cs, yte_cs):.4f}')
print(f'Depth-3    — depth=3, '
      f'train acc={tree_d3.score(Xtr_cs, ytr_cs):.4f}, '
      f'test acc={tree_d3.score(Xte_cs, yte_cs):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(tree_d3, feature_names=list(X_cs.columns),
          class_names=['Low', 'High'], filled=True, ax=ax, fontsize=8)
ax.set_title('Depth-3 classification tree on Carseats — predict High sales')
plt.show()

### 2.3 Heart disease dataset — full tree

In [ ]:
# Encode heart data
h_heart = heart.copy()
# Encode categorical columns
cat_heart = h_heart.select_dtypes(include='object').columns.tolist()
cat_heart = [c for c in cat_heart if c != 'AHD']
h_heart = pd.get_dummies(h_heart, columns=cat_heart, drop_first=True, dtype=float)
h_heart['AHD'] = (h_heart['AHD'] == 'Yes').astype(int)

X_hrt = h_heart.drop(columns=['AHD'])
y_hrt = h_heart['AHD']
Xtr_hrt, Xte_hrt, ytr_hrt, yte_hrt = train_test_split(
    X_hrt, y_hrt, test_size=0.25, random_state=0, stratify=y_hrt)

tree_hrt = DecisionTreeClassifier(random_state=0).fit(Xtr_hrt, ytr_hrt)
print(f'Heart tree — depth={tree_hrt.get_depth()}, '
      f'train acc={tree_hrt.score(Xtr_hrt, ytr_hrt):.4f}, '
      f'test acc={tree_hrt.score(Xte_hrt, yte_hrt):.4f}')

In [ ]:
# Train / test error as a function of tree depth (like ISLR Figure 8.5)
depths = range(1, 16)
train_err, test_err = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr_hrt, ytr_hrt)
    train_err.append(1 - m.score(Xtr_hrt, ytr_hrt))
    test_err.append(1 - m.score(Xte_hrt, yte_hrt))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_err, marker='.', label='Train error')
ax.plot(depths, test_err,  marker='.', label='Test error')
ax.set_xlabel('Tree depth'); ax.set_ylabel('Error rate')
ax.set_title('Heart: classification error vs tree depth')
ax.legend(); plt.show()
print(f'Best test error = {min(test_err):.4f} at depth {depths[int(np.argmin(test_err))]}')

---
## 3. Cost-Complexity Pruning

Growing the full tree overfits. Rather than setting `max_depth` by hand, we grow the full tree then **prune** it.

### 3.1 The pruned-subtree criterion

For a subtree $T$ of the full tree $T_0$, define the penalised cost:

$$C_\alpha(T) = \sum_{m=1}^{|T|} \sum_{i:\, x_i \in R_m} (y_i - \hat{y}_{R_m})^2 + \alpha |T|$$

where $|T|$ is the number of terminal nodes (leaves) and $\alpha \geq 0$ is a **complexity parameter**.

- $\alpha = 0$: no penalty → full tree
- $\alpha \to \infty$: penalty crushes all splits → root-only stump
- **Weakest-link pruning** finds the sequence of nested subtrees $T_0 \supset T_1 \supset \ldots$ each minimising $C_\alpha$ for some range of $\alpha$.

We select the best $\alpha$ by **K-fold cross-validation**.

In [ ]:
# ── Regression tree pruning on Hitters ──────────────────────────────────────
full_reg2 = DecisionTreeRegressor(random_state=0).fit(Xtr, ytr)
path = full_reg2.cost_complexity_pruning_path(Xtr, ytr)
alphas = path.ccp_alphas[:-1]  # drop root-only stump

kf = KFold(n_splits=5, shuffle=True, random_state=0)
cv_mse, cv_std = [], []
for a in alphas:
    scores = -cross_val_score(DecisionTreeRegressor(random_state=0, ccp_alpha=a),
                               Xtr, ytr, cv=kf, scoring='neg_mean_squared_error')
    cv_mse.append(scores.mean())
    cv_std.append(scores.std())

best_a = alphas[int(np.argmin(cv_mse))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(alphas, cv_mse, yerr=cv_std, fmt='.', capsize=3, alpha=0.7)
ax.axvline(best_a, color='C3', linestyle='--', label=f'best α = {best_a:.4f}')
ax.set_xlabel(r'$\alpha$ (complexity penalty)')
ax.set_ylabel('5-fold CV MSE')
ax.set_title('Regression tree: cost-complexity pruning path (Hitters)')
ax.legend(); plt.show()

pruned_reg = DecisionTreeRegressor(random_state=0, ccp_alpha=best_a).fit(Xtr, ytr)
print(f'Full tree  — depth={full_reg2.get_depth()}, leaves={full_reg2.get_n_leaves()}')
print(f'Pruned     — depth={pruned_reg.get_depth()}, leaves={pruned_reg.get_n_leaves()}')
print(f'Full   test MSE = {mean_squared_error(yte, full_reg2.predict(Xte)):.4f}')
print(f'Pruned test MSE = {mean_squared_error(yte, pruned_reg.predict(Xte)):.4f}')

In [ ]:
# ── Classification tree pruning on Heart ────────────────────────────────────
full_hrt2 = DecisionTreeClassifier(random_state=0).fit(Xtr_hrt, ytr_hrt)
path_hrt = full_hrt2.cost_complexity_pruning_path(Xtr_hrt, ytr_hrt)
alphas_hrt = path_hrt.ccp_alphas[:-1]

cv_acc = []
for a in alphas_hrt:
    scores = cross_val_score(DecisionTreeClassifier(random_state=0, ccp_alpha=a),
                              Xtr_hrt, ytr_hrt, cv=5)
    cv_acc.append(scores.mean())

best_a_hrt = alphas_hrt[int(np.argmax(cv_acc))]
pruned_hrt = DecisionTreeClassifier(random_state=0, ccp_alpha=best_a_hrt).fit(Xtr_hrt, ytr_hrt)

# Side-by-side: pruned vs full (depth-capped for readability)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_tree(pruned_hrt, feature_names=list(X_hrt.columns),
          class_names=['No', 'Yes'], filled=True, ax=axes[0], fontsize=7)
axes[0].set_title(f'Pruned (α={best_a_hrt:.4f}, depth={pruned_hrt.get_depth()})')

cap = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr_hrt, ytr_hrt)
plot_tree(cap, feature_names=list(X_hrt.columns),
          class_names=['No', 'Yes'], filled=True, ax=axes[1], fontsize=7)
axes[1].set_title('Depth-3 (hand-capped, for comparison)')
plt.tight_layout(); plt.show()

print(f'Full   test acc = {full_hrt2.score(Xte_hrt, yte_hrt):.4f}')
print(f'Pruned test acc = {pruned_hrt.score(Xte_hrt, yte_hrt):.4f}')
print(f'Cap d3 test acc = {cap.score(Xte_hrt, yte_hrt):.4f}')

---
## 4. Trees vs Linear Models

Neither decision trees nor linear models dominate the other — the better choice depends on the **true decision boundary** (ISLR Figure 8.7).

- If the true boundary is **linear**, logistic regression wins: trees waste splits approximating a straight line with staircase steps.
- If the true boundary is **non-linear or axis-aligned**, trees win: they carve out exactly the relevant region in a few splits.

In [ ]:
def plot_boundary(ax, model, X, y, title):
    x1_min, x1_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    x2_min, x2_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                            np.linspace(x2_min, x2_max, 200))
    Z = model.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)
    ax.contourf(xx1, xx2, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm',
               edgecolors='k', linewidths=0.4, s=25)
    ax.set_title(title)

# Scenario A: linear boundary
n = 300
Xa = rng.normal(0, 1, (n, 2))
ya = (Xa[:, 0] + Xa[:, 1] > 0).astype(int)

# Scenario B: non-linear (checkerboard) boundary
Xb = rng.uniform(-2, 2, (n, 2))
yb = ((np.abs(Xb[:, 0]) < 1) == (np.abs(Xb[:, 1]) < 1)).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, (X_sc, y_sc, label) in enumerate([(Xa, ya, 'Linear boundary'),
                                           (Xb, yb, 'Non-linear boundary')]):
    lr = LogisticRegression().fit(X_sc, y_sc)
    dt = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_sc, y_sc)
    plot_boundary(axes[i, 0], lr, X_sc, y_sc,
                  f'{label}\nLogistic Reg acc={lr.score(X_sc, y_sc):.3f}')
    plot_boundary(axes[i, 1], dt, X_sc, y_sc,
                  f'{label}\nDecision Tree acc={dt.score(X_sc, y_sc):.3f}')
plt.suptitle('Trees vs Linear Models: when each wins', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## 5. Strengths, Weaknesses & Variance Demo

### Strengths
- **Interpretable:** every prediction is a flowchart — you can explain it to a non-technical stakeholder
- **No feature scaling needed:** splits are based on ordering, not distances
- **Handles mixed types naturally:** numeric and categorical features in the same model
- **Captures interactions automatically:** a split on `X1` within a subtree already conditioned on `X2`
- **Mirrors human reasoning:** "if Years < 4.5 and Hits < 117.5, predict low salary"

### Weaknesses
- **High variance:** small changes in the training data → completely different tree structure
- **Axis-aligned splits only:** can't approximate a diagonal boundary without many splits
- **Greedy:** each split is locally optimal but not globally
- **Not competitive:** a single tree rarely matches linear models on linear data or ensembles on complex data

The demo below shows the variance problem directly: fit four trees to four bootstrap samples of the same data and watch the splits change completely.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, seed in zip(axes.ravel(), [0, 1, 2, 3]):
    rng_b = np.random.default_rng(seed)
    idx = rng_b.choice(len(Xtr_cs), len(Xtr_cs), replace=True)
    t = DecisionTreeClassifier(max_depth=3, random_state=seed)
    t.fit(Xtr_cs.iloc[idx], ytr_cs.iloc[idx])
    plot_tree(t, feature_names=list(X_cs.columns),
              class_names=['Low', 'High'], filled=True, ax=ax, fontsize=6)
    ax.set_title(f'Bootstrap sample #{seed+1} — root: {X_cs.columns[t.tree_.feature[0]]}')
plt.suptitle('Same data, 4 bootstrap samples → 4 different trees (variance!)',
             fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

Same data, different random samples — the root split changes each time, and the tree structure is completely different.

**This variance is the problem that ensembles solve (Course 8).**  
- **Bagging:** average B high-variance trees → lower variance
- **Random Forests:** decorrelate the trees before averaging → even lower variance
- **Boosting:** instead of averaging, build a sequence of trees each correcting the errors of the previous

---

# Exercises

Each exercise has a blank starter cell followed by a solution. Try each before revealing the solution.

---
## Exercise 1

**Task.** Using the **Hitters** dataset, fit a regression tree on ALL features (not just Years and Hits) to predict `log(Salary)`.  
Apply cost-complexity pruning with 5-fold CV.  
Report the best $\alpha$, the number of leaves before and after pruning, and the test MSE before and after pruning.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np, pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np, pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

hitters = load_dataset('hitters')
h = hitters.dropna(subset=['Salary']).copy()
h['LogSalary'] = np.log(h['Salary'])
h_enc = pd.get_dummies(h.drop(columns=['Salary']),
                        columns=['League', 'Division', 'NewLeague'],
                        drop_first=True, dtype=float)
X = h_enc.drop(columns=['LogSalary'])
y = h_enc['LogSalary']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

full = DecisionTreeRegressor(random_state=0).fit(Xtr, ytr)
path = full.cost_complexity_pruning_path(Xtr, ytr)
alphas = path.ccp_alphas[:-1]
cv_scores = [-cross_val_score(DecisionTreeRegressor(random_state=0, ccp_alpha=a),
                               Xtr, ytr, cv=5, scoring='neg_mean_squared_error').mean()
             for a in alphas]
best_a = alphas[int(np.argmin(cv_scores))]
pruned = DecisionTreeRegressor(random_state=0, ccp_alpha=best_a).fit(Xtr, ytr)

print(f'Best α = {best_a:.5f}')
print(f'Full   — leaves={full.get_n_leaves()},   test MSE={mean_squared_error(yte, full.predict(Xte)):.4f}')
print(f'Pruned — leaves={pruned.get_n_leaves()}, test MSE={mean_squared_error(yte, pruned.predict(Xte)):.4f}')

---
## Exercise 2

**Task.** Using the **Heart** dataset (loaded above), fit a `DecisionTreeClassifier`.  
1. Report train and test accuracy.  
2. Plot the confusion matrix on the test set.
3. Which feature is at the root split?

In [ ]:
# your code here (heart is already loaded above as X_hrt, y_hrt)


### Exercise 2 — Solution

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

tree_ex2 = DecisionTreeClassifier(random_state=0).fit(Xtr_hrt, ytr_hrt)
print(f'Train acc = {tree_ex2.score(Xtr_hrt, ytr_hrt):.4f}')
print(f'Test  acc = {tree_ex2.score(Xte_hrt, yte_hrt):.4f}')
print(f'Root split: {X_hrt.columns[tree_ex2.tree_.feature[0]]}')

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_estimator(tree_ex2, Xte_hrt, yte_hrt,
                                       display_labels=['No AHD', 'AHD'], ax=ax)
ax.set_title('Heart: confusion matrix (test set)')
plt.show()

---
## Exercise 3

**Task.** On the **Carseats** `High` classification problem, plot train and test accuracy as a function of `max_depth` (1 to 15).  
Then apply cost-complexity pruning and add the pruned tree's test accuracy as a horizontal line.  
At which depth does overfitting begin?

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

depths = range(1, 16)
tr_acc, te_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr_cs, ytr_cs)
    tr_acc.append(m.score(Xtr_cs, ytr_cs))
    te_acc.append(m.score(Xte_cs, yte_cs))

# Pruned tree
full_cs = DecisionTreeClassifier(random_state=0).fit(Xtr_cs, ytr_cs)
path_cs = full_cs.cost_complexity_pruning_path(Xtr_cs, ytr_cs)
cv_cs   = [cross_val_score(DecisionTreeClassifier(random_state=0, ccp_alpha=a),
                            Xtr_cs, ytr_cs, cv=5).mean()
           for a in path_cs.ccp_alphas[:-1]]
best_a_cs  = path_cs.ccp_alphas[:-1][int(np.argmax(cv_cs))]
pruned_cs  = DecisionTreeClassifier(random_state=0, ccp_alpha=best_a_cs).fit(Xtr_cs, ytr_cs)
pruned_acc = pruned_cs.score(Xte_cs, yte_cs)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, tr_acc, marker='.', label='Train accuracy')
ax.plot(depths, te_acc, marker='.', label='Test accuracy')
ax.axhline(pruned_acc, color='C3', linestyle='--', lw=1.5,
           label=f'Pruned test acc = {pruned_acc:.4f}')
ax.set_xlabel('max_depth'); ax.set_ylabel('Accuracy')
ax.set_title('Carseats: depth vs accuracy (overfitting visible past ~depth 4)')
ax.legend(); plt.show()
print(f'Overfitting begins around depth {int(np.argmin(te_acc[:8]))+1}')

---
## Exercise 4

**Task.** Recreate the **Years / Hits partition plot** (similar to ISLR Figure 8.1) but for a depth-3 tree instead of depth-2.  
Colour the regions by predicted log(Salary), overlay the data points, and draw the split boundary lines.

In [ ]:
# your code here


### Exercise 4 — Solution

In [ ]:
from shared.data_utils import load_dataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

hitters = load_dataset('hitters')
h = hitters.dropna(subset=['Salary']).copy()
h['LogSalary'] = np.log(h['Salary'])

tree3 = DecisionTreeRegressor(max_depth=3, random_state=0)
tree3.fit(h[['Years', 'Hits']], h['LogSalary'])

yg = np.linspace(h['Years'].min() - 0.5, h['Years'].max() + 0.5, 400)
hg = np.linspace(h['Hits'].min() - 5,   h['Hits'].max() + 5,   400)
YY, HH = np.meshgrid(yg, hg)
Z = tree3.predict(np.c_[YY.ravel(), HH.ravel()]).reshape(YY.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(YY, HH, Z, levels=30, cmap='YlOrRd', alpha=0.65)
ax.contour(YY, HH, Z, levels=30, colors='k', linewidths=0.3, alpha=0.3)
sc = ax.scatter(h['Years'], h['Hits'], c=h['LogSalary'],
                cmap='YlOrRd', edgecolors='k', linewidths=0.4, s=30, zorder=3)
plt.colorbar(sc, ax=ax, label='log(Salary)')
ax.set_xlabel('Years in MLB'); ax.set_ylabel('Hits (previous year)')
ax.set_title('Depth-3 tree partition — Hitters log(Salary)')
plt.show()